# Full-match inference on Colab T4

Run a full match (90+ minutes) through the CV pipeline on a GPU
runtime. Outputs the `analysis-stub` pickle that the desktop event
tagger ingests for per-player + per-event analytics.

## Before you start

1. **Runtime → Change runtime type → T4 GPU**.
2. Put these in a single folder on your Google Drive (e.g.
   `MyDrive/football_analytics/`):
   * `videos/<match>.mp4` — the source video
   * `models/best_v7.pt` — your trained weights
   * `calibrations/<match>.json` — per-clip calibration JSON (optional but
     produces world coords + speed/distance metrics)
3. Edit **Cell 4 (Paths)** so `MATCH_NAME` matches your video file's stem.

## After it finishes

Cell 6 saves the analysis stub to your Drive. Download it to your
laptop, then run locally:

```
python scripts/ingest_match.py \
    --db data/analytics.db \
    --stub stubs/<match>_pass1.pkl \
    --video video_clips/<match>.mp4 \
    --calibration calibrations/<match>.json \
    --model-version v7 \
    --club "..." --season "..." \
    --home-team "..." --away-team "..." \
    --match-date YYYY-MM-DD \
    --create-missing
```

Then open the app: `python -m src.analytics.app --db data/analytics.db`.

In [ ]:
# 1. Confirm GPU + mount Drive.
import torch
assert torch.cuda.is_available(), (
    'No GPU. Runtime menu → Change runtime type → T4 GPU.'
)
print(f'GPU: {torch.cuda.get_device_name(0)}')

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Install dependencies. opencv-python-headless avoids the GUI libs
# that don't exist on Colab; everything else mirrors the laptop venv.
!pip install -q ultralytics supervision opencv-python-headless \
    scikit-learn matplotlib tqdm scipy

In [ ]:
# 3. Clone the repo at the analytics-platform branch. Cells 5/6 below
# only need main.py + the src/ tree, so a shallow clone is plenty.
import os
REPO_DIR = '/content/football_analysis_tool'
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 -b analytics-platform \
        https://github.com/tadpau/football_analysis_tool.git $REPO_DIR
%cd $REPO_DIR

In [ ]:
# 4. Paths — edit MATCH_NAME and DRIVE_ROOT for your setup.
# Everything downstream is derived from these so you don't have to
# remember 7 different filenames.
DRIVE_ROOT  = '/content/drive/MyDrive/football_analytics'
MATCH_NAME  = 'VFA_KFM_full_match'     # ← change me. Stem of your .mp4.
MODEL_FILE  = 'best_v7.pt'

VIDEO_PATH  = f'{DRIVE_ROOT}/videos/{MATCH_NAME}.mp4'
MODEL_PATH  = f'{DRIVE_ROOT}/models/{MODEL_FILE}'
CALIB_PATH  = f'{DRIVE_ROOT}/calibrations/{MATCH_NAME}.json'
STUB_PATH   = f'{DRIVE_ROOT}/stubs/{MATCH_NAME}_v7_pass1.pkl'

os.makedirs(f'{DRIVE_ROOT}/stubs', exist_ok=True)

for p, name in [(VIDEO_PATH, 'video'), (MODEL_PATH, 'model')]:
    assert os.path.exists(p), f'{name} not found at {p} — upload it to Drive first.'
has_calib = os.path.exists(CALIB_PATH)
print(f'video       : {VIDEO_PATH}')
print(f'model       : {MODEL_PATH}')
print(f'calibration : {CALIB_PATH if has_calib else "(none — world coords will be NULL)"}')
print(f'stub output : {STUB_PATH}')

In [ ]:
# 5. Run pass-1 inference only. The desktop event-tagger draws overlays
# from the DB, so the rendered MP4 (pass 2) is wasted compute for a
# full-match Colab run — --skip-render cuts wall-clock roughly in half.
#
# On a T4 at imgsz=640 expect ~45 fps pass-1 throughput; a 2-hour
# match completes in ~80 minutes. imgsz=1280 doubles that but gives
# slightly better detection of small/far players + ball. Pick based
# on your Colab session-length comfort — 640 is the safe default for
# free-tier sessions.

calib_flag = f'--calibration "{CALIB_PATH}"' if has_calib else ''

!python main.py \
    --input "$VIDEO_PATH" \
    --model "$MODEL_PATH" --mode custom_landmarks \
    --stream --imgsz 640 --chunk-size 800 \
    $calib_flag \
    --analysis-stub "$STUB_PATH" \
    --skip-render

In [ ]:
# 6. Verify the stub landed on Drive + show its size.
import os
if os.path.exists(STUB_PATH):
    size_mb = os.path.getsize(STUB_PATH) / (1024 * 1024)
    print(f'✓ stub saved: {STUB_PATH}  ({size_mb:.1f} MB)')
    print()
    print('Download this file to your laptop, then run:')
    print(f'  scripts/ingest_match.py --stub stubs/{os.path.basename(STUB_PATH)} ...')
else:
    print('✗ stub missing — check the cell-5 output for errors.')